**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Uncertainty in ML

A model that says '87%' should be right 87% of the time — most deep models aren't. Two sessions on measuring calibration and fixing it with the workhorse tools: temperature scaling and deep ensembles, with a hard look at what happens *off*-distribution.

## 1. Pre-requisites

- [Training Dynamics](./Training_Dynamics.ipynb) (we reuse its spiral testbed).
- [Kernel Methods](./Kernel_Methods.ipynb) S2 — GPs as the calibration gold standard.
- [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) S3 — the Bayesian frame.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# noisy spirals again — genuine class overlap means genuine aleatoric uncertainty
def spirals(n=3000, noise=0.9, seed=0):
    r = np.random.default_rng(seed)
    t = np.linspace(0.5, 3*np.pi, n//2)
    X, y = [], []
    for cls, ph in [(0, 0.0), (1, np.pi)]:
        X.append(np.stack([t*np.cos(t+ph), t*np.sin(t+ph)], 1) + noise*r.standard_normal((n//2, 2)))
        y.append(np.full(n//2, cls))
    X = np.concatenate(X).astype(np.float32); y = np.concatenate(y).astype(np.int64)
    X = (X - X.mean(0)) / X.std(0)
    idx = r.permutation(n)
    return torch.from_numpy(X[idx]), torch.from_numpy(y[idx])

Xa, ya = spirals()
Xtr, ytr, Xte, yte = Xa[:2000], ya[:2000], Xa[2000:], ya[2000:]

def make_net(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 2))

def fit(net, epochs=600):     # deliberately overtrained — watch the confidence outrun the accuracy
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)
    for ep in range(epochs):
        for i in range(0, 2000, 200):
            opt.zero_grad()
            Fn.cross_entropy(net(Xtr[i:i+200]), ytr[i:i+200]).backward()
            opt.step()
    return net

---
### 🕐 Session 1 of 2 — *Calibration & Temperature Scaling* (~40 min)
**Goal:** measure whether confidences mean anything; fix miscalibration with one parameter.
**Builds on:** [Training Dynamics](./Training_Dynamics.ipynb). &nbsp; **Feeds into:** Session 2 (ensembles & OOD).

---

## 2. Does 87% Mean 87%?

💡 **Intuition.** Accuracy asks 'how often right?'; **calibration** asks 'when you say 87%, are you right 87% of the time?' The reliability diagram answers it: bin predictions by confidence, plot accuracy per bin against the diagonal. Deep nets trained to convergence sit *below* the diagonal — overconfident — because cross-entropy keeps rewarding sharper probabilities long after accuracy saturates. The embarrassingly effective fix: **temperature scaling** — divide the logits by one scalar $T$ fitted on validation data. It can't change any decision (argmax is $T$-invariant); it only makes the *confidence honest*.

In [ ]:

# YOUR CODE HERE


In [ ]:
# fit T on a validation split by minimizing NLL — one parameter, no retraining

# YOUR CODE HERE


---
### 🕐 Session 2 of 2 — *Ensembles & the Out-of-Distribution Problem* (~40 min)
**Goal:** average independently-trained nets for better uncertainty; test where all bets are off.
**Builds on:** Session 1.

---

## 3. Deep Ensembles

💡 **Intuition.** Train the same architecture from $k$ different random seeds and *average the probabilities*. Where the data constrains the function, the members agree; where it doesn't, they disagree — and that **disagreement is an uncertainty signal** that single-model confidence simply doesn't carry. It's a crude Bayesian posterior ([Estimation Theory S3](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb)), cousin to the [particle filter](../Intro_Time_Series/Beyond_Kalman.ipynb): represent belief with samples. Cost: $k\times$ everything — and it remains the strongest practical baseline in the field.

In [ ]:

# YOUR CODE HERE


## 4. Off the Map

💡 **Intuition.** The dirty secret of every confidence score: it is only meaningful **on the distribution the model was trained on**. Show the network a point from a different world and softmax still prints a confident number — softmax *must* sum to one; it has no 'none of the above'. Ensemble disagreement (predictive entropy) at least *rises* off-distribution. Compare both on points far outside the spiral:

In [ ]:
# scan the plane: single-model confidence vs ensemble entropy
# find the off-map points the ensemble actually flags (and admit the ones it doesn't)

# YOUR CODE HERE


## 5. Conclusion

Calibrate before you trust (temperature is nearly free); ensemble when the stakes justify 5×; and treat *all* confidences as conditional on being on-distribution — detecting 'off the map' is its own problem, and disagreement is your first tool — but as the area numbers show, it flags only *part* of the off-map world: ensembles mitigate OOD overconfidence, they do not solve it. The [GP](./Kernel_Methods.ipynb) remains the standard these methods chase.

---
## Where next

- [Kernel Methods](./Kernel_Methods.ipynb) S2 — calibrated uncertainty by construction.
- [Beyond Kalman](../Intro_Time_Series/Beyond_Kalman.ipynb) — belief-as-samples, the filtering version.
- [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) — what 'well-calibrated' means formally.